https://msea.readthedocs.io/en/latest/tutorial.html

In [1]:
import msea
# %pip install msea
from msea import SetLibrary
import numpy
import scipy
import pandas as pd
from pprint import pprint
# %pip install biom-format
from biom import load_table  # reqired for parsing BIOM formated dataset
from skbio.stats.composition import ancom
# %pip install scikit-bio
import re


- read the file
- split into counts and metadata

In [2]:
# microbiome_data = pd.read_excel("../csv_files/Xero-Microbiome RA data patient matched AH 07 23 25 females.xlsx")
microbiome_data = pd.read_excel("../csv_files/Xero-Microbiome RA data patient matched AH 07 23 25 females saliva.xlsx")

cols = list(microbiome_data.columns)
microbiome_data.reindex(columns=cols[-3:]+cols[:-3]) ## Group, Condition, Site

microbiome_data = microbiome_data.drop("...491", axis = 1)

cols = list(microbiome_data.columns)
metadata = microbiome_data[ cols[0:4] ]

data_cols = microbiome_data[ cols[5:-1] ]

## aggregate to the genus level
data_cols.columns = [col.split(";")[-2] if "g__" in col else col for col in data_cols.columns]
data_cols = data_cols.T.groupby(data_cols.columns).sum().T

# data_cols = data_cols.loc[:,["g__Porphyromonas", "g__Haemophilus", "g__Streptococcus"]]
data_cols = data_cols.loc[:,["g__Porphyromonas", "g__Haemophilus", "g__Streptococcus", "g__Rothia"]]
data_cols




,g__Porphyromonas,g__Haemophilus,g__Streptococcus,g__Rothia
0,0.018009,0.037097,0.222778,0.077264
1,0.046521,0.021724,0.190622,0.072701
2,0.014745,0.031458,0.281777,0.015183
3,0.001438,0.014341,0.362370,0.199478
4,0.050729,0.013685,0.235281,0.047381
5,0.060231,0.032437,0.318238,0.106341
6,0.052619,0.028529,0.327879,0.108863
7,0.000289,0.008694,0.317691,0.116719
8,0.011597,0.056673,0.132547,0.041958
9,0.003558,0.025730,0.315425,0.022603


Next, we perform DA analysis using ANCOM to identify DA microbes:

In [3]:
ancom_df, percentile_df = ancom(data_cols + 1,  # adding pseudocounts
                                metadata['Group'],
                                alpha=0.1,
                                p_adjust='holm-bonferroni')

microbes_DA = ancom_df.loc[ancom_df['Signif']].index

ancom_df.loc[ancom_df['Signif']].to_csv("../output/without_men_saliva/msea_significantRF/significant_bacteria.csv") ## save the ancom_df

print(microbes_DA)

# retain genus names
microbes_DA = filter(None, [s.split('; ')[-1].lstrip('g__')
                            for s in microbes_DA])
# convert to set
microbes_DA = set(microbes_DA)


percentile_df

Index(['g__Streptococcus'], dtype='object')


Percentile,0.0,25.0,50.0,75.0,100.0,0.0,25.0,50.0,75.0,100.0
Group,Control,Control,Control,Control,Control,Xerostomic,Xerostomic,Xerostomic,Xerostomic,Xerostomic
g__Porphyromonas,1.002611,1.017940,1.024515,1.043084,1.063382,1.000000,1.000490,1.009479,1.020163,1.060231
g__Haemophilus,1.011205,1.027111,1.039937,1.048341,1.155559,1.001714,1.010929,1.022541,1.032192,1.067910
g__Streptococcus,1.159289,1.238113,1.277269,1.286847,1.491980,1.132547,1.276086,1.317918,1.355346,1.651164
g__Rothia,1.025695,1.089792,1.199637,1.277262,1.416601,1.015183,1.065730,1.107602,1.153819,1.313001


Finally, we can perform MSEA for DA microbes we just identified:

In [4]:
gmt_filepath = \
    'https://bitbucket.org/wangz10/msea/raw/aee6dd184e9bde152b4d7c2f3c7245efc1b80d23/msea/data/human_genes_associated_microbes/set_library.gmt'

d_gmt = msea.read_gmt(gmt_filepath)

# Look at a couple of reference sets in the library
pprint(list(d_gmt.items())[:3])

[('A2M', {'Pseudomonas', 'Azomonas', 'Sodalis', 'Borrelia', 'Salmonella'}),
 ('AAAS',
  {'Colwellia',
   'Deinococcus',
   'Idiomarina',
   'Neisseria',
   'Pseudidiomarina',
   'Pseudoalteromonas'}),
 ('AACS',
  {'Acetobacter',
   'Acinetobacter',
   'Azomonas',
   'Corynebacterium',
   'Enterobacter',
   'Klebsiella',
   'Mycobacterium',
   'Mycoplasma',
   'Pseudomonas',
   'Sodalis',
   'Staphylococcus',
   'Streptomyces',
   'Tetragenococcus'})]


In [5]:
# this can be done using the `msea.enrich` function
msea_result = msea.enrich(microbes_DA, d_gmt=d_gmt, universe=1000)
# check the top enriched reference microbe-sets
print(msea_result.head())

        oddsratio    pvalue    qvalue           shared  n_shared
term                                                            
MBL2    55.555556  0.036926  0.222081  [Streptococcus]         1
MOG     52.631579  0.038812  0.222081  [Streptococcus]         1
MN1     52.631579  0.038812  0.222081  [Streptococcus]         1
MAPK1  111.111111  0.019694  0.222081  [Streptococcus]         1
CTSC   142.857143  0.015802  0.222081  [Streptococcus]         1


Step 2: perform MSEA with adjustment of expected ranks for reference sets. Sometimes certain reference microbe-sets in a library are more likely to be enriched by chance. We can adjust this by empirically estimating the null distributions of the ranks of the reference sets, then uses z-score to quantify if observed ranks are significantly different from the expected ranks.

To do that, it is easier through the [`SetLibrary`](https://msea.readthedocs.io/en/latest/api_docs.html#msea.SetLibrary) class:

The [`SetLibrary.get_empirical_ranks()`](https://msea.readthedocs.io/en/latest/api_docs.html#msea.SetLibrary.get_empirical_ranks) method helps compute the expected ranks and store the means and standard deviations of the ranks from the null distribution:


In [6]:
set_library = SetLibrary.load(gmt_filepath)
set_library.get_empirical_ranks(n=20)
print(set_library.rank_means.shape, set_library.rank_stds.shape)



Calculating empirical ranks for each set...
Number of unique microbes: 566


100%|███████████████████████████████████████████████████████████████████████████████████| 20/20 [00:06<00:00,  3.22it/s]

(1286,) (1286,)


After that, we can perform MSEA with this adjustment:


In [7]:

msea_result_adj = set_library.enrich(
    microbes_DA, adjust=True, universe=1000)
print(msea_result_adj.head())

msea_result_adj

        oddsratio    pvalue    qvalue    zscore  combined_score  \
term                                                              
OCLN    35.714286  0.055545  0.226045 -6.373168       18.422087   
TREM2   71.428571  0.029324  0.222081 -4.522584       15.961788   
SP3     83.333333  0.025489  0.222081 -4.302712       15.788811   
PYY     76.923077  0.027409  0.222081 -4.232912       15.225234   
TNN    142.857143  0.015802  0.222081 -3.229284       13.393791   

                shared  n_shared  
term                              
OCLN   [Streptococcus]         1  
TREM2  [Streptococcus]         1  
SP3    [Streptococcus]         1  
PYY    [Streptococcus]         1  
TNN    [Streptococcus]         1  


,oddsratio,pvalue,qvalue,zscore,combined_score,shared,n_shared
term,,,,,,,
OCLN,35.714286,0.055545,0.226045,-6.373168,18.422087,[Streptococcus],1
TREM2,71.428571,0.029324,0.222081,-4.522584,15.961788,[Streptococcus],1
SP3,83.333333,0.025489,0.222081,-4.302712,15.788811,[Streptococcus],1
PYY,76.923077,0.027409,0.222081,-4.232912,15.225234,[Streptococcus],1
TNN,142.857143,0.015802,0.222081,-3.229284,13.393791,[Streptococcus],1
...,...,...,...,...,...,...,...
FCN2,83.333333,0.025489,0.222081,3.399011,-12.472679,[Streptococcus],1
CHIT1,166.666667,0.013848,0.222081,2.917187,-12.484540,[Streptococcus],1
FASLG,47.619048,0.042569,0.222081,3.993008,-12.604458,[Streptococcus],1


In [8]:
msea_result_adj.to_csv("../output/without_men_saliva/msea_significantRF/msea_results.csv")